*Autor: mCárdenas 2026*

## 1. Instalación de la librería redis-py

Asegúrate de instalar la librería antes de ejecutar el script:

In [ ]:
# docker run -d --name redis-server -p 6379:6379 -v redis-data:/data redis

In [ ]:
#!pip install redis

## 2. Script completo

Este script contiene tanto el Publisher como el Subscriber en dos funciones separadas. Puedes ejecutarlos en procesos separados para que interactúen.

In [ ]:
import redis
import time
import threading

# Conexión al servidor Redis
redis_client = redis.StrictRedis(host='localhost', port=6379, decode_responses=True)

# Función para el Subscriber
def subscriber(channel_name):
    pubsub = redis_client.pubsub()  # Crear el objeto pubsub
    pubsub.subscribe(channel_name)  # Suscribirse al canal
    
    print(f"Subscriber: Suscrito al canal '{channel_name}'")
    
    for message in pubsub.listen():
        if message["type"] == "message":  # Ignorar otros eventos como "subscribe"
            print(f"Subscriber: Recibido -> {message['data']}")

# Función para el Publisher
def publisher(channel_name):
    print(f"Publisher: Publicando en el canal '{channel_name}'")
    count = 1
    
    while True:
        message = f"Mensaje #{count}"
        redis_client.publish(channel_name, message)  # Publicar un mensaje en el canal
        print(f"Publisher: Enviado -> {message}")
        count += 1
        time.sleep(2)  # Esperar 2 segundos entre mensajes

# Canal de comunicación
channel = "demo_channel"

# Crear dos hilos: uno para el Publisher y otro para el Subscriber
thread_subscriber = threading.Thread(target=subscriber, args=(channel,))
thread_publisher = threading.Thread(target=publisher, args=(channel,))

# Iniciar los hilos
thread_subscriber.start()
thread_publisher.start()

# Esperar a que los hilos terminen (esto no ocurrirá porque el script es infinito)
thread_subscriber.join()
thread_publisher.join()


#### Explicación del Script:

- Conexión a Redis: el script utiliza redis.StrictRedis para conectarse al servidor Redis. Por defecto, se conecta al localhost y al puerto 6379.

*Subscriber*:

- Se crea un objeto pubsub para escuchar los mensajes del canal.
- Se suscribe al canal demo_channel y escucha mensajes en un bucle infinito con pubsub.listen().
- Filtra los mensajes reales con if message["type"] == "message".

*Publisher*:
- Publica mensajes en el canal demo_channel en un bucle infinito.
- Utiliza redis_client.publish para enviar un mensaje al canal.

*Hilos (Threads)*:
- Para que Publisher y Subscriber se ejecuten simultáneamente en el mismo script, se usan hilos con la librería threading.

*Canal*:
- Se define el canal como "demo_channel", pero puedes cambiarlo por cualquier nombre.

#### Cómo ejecutar el script:

- Opción 1: Ejecuta el script tal como está. Ambos roles (Publisher y Subscriber) funcionarán en hilos diferentes.
- Opción 2: Separa el Publisher y el Subscriber en dos scripts independientes y ejecútalos en terminales diferentes.

## Script para el Subscriber (separado):

In [ ]:
import redis

def subscriber(channel_name):
    redis_client = redis.StrictRedis(host='localhost', port=6379, decode_responses=True)
    pubsub = redis_client.pubsub()
    pubsub.subscribe(channel_name)
    print(f"Subscriber: Suscrito al canal '{channel_name}'")
    for message in pubsub.listen():
        if message["type"] == "message":
            print(f"Subscriber: Recibido -> {message['data']}")

subscriber("demo_channel")

## Script para el Publisher (separado):

In [ ]:
import redis
import time

def publisher(channel_name):
    redis_client = redis.StrictRedis(host='localhost', port=6379, decode_responses=True)
    print(f"Publisher: Publicando en el canal '{channel_name}'")
    count = 1
    while True:
        message = f"Mensaje #{count}"
        redis_client.publish(channel_name, message)
        print(f"Publisher: Enviado -> {message}")
        count += 1
        time.sleep(2)

publisher("demo_channel")

<pre>
Ejemplo de salida:

Publisher:

Publisher: Publicando en el canal 'demo_channel'
Publisher: Enviado -> Mensaje #1
Publisher: Enviado -> Mensaje #2
Publisher: Enviado -> Mensaje #3
Subscriber:

Subscriber: Suscrito al canal 'demo_channel'
Subscriber: Recibido -> Mensaje #1
Subscriber: Recibido -> Mensaje #2
Subscriber: Recibido -> Mensaje #3
</pre>